# DLK Lyrik Korpus - Vektordatenbank mit Chroma

Dieses Notebook liest den Deutschen Lyrik Korpus (DLK) aus JSON-Dateien ein und embeddet die Gedichte in eine Chroma-Vektordatenbank.

In [ ]:
%pip install chromadb sentence-transformers tqdm

In [ ]:
import os
import json
import glob
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions

In [ ]:
# -------------------------
# Konfiguration
# -------------------------
DLK_PATH = r'D:\temp\lyrik\DLK\DLK\meterized\json_DLK_v6'
CHROMA_PATH = './chroma_dlk_db'
COLLECTION_NAME = 'dlk_lyrik'
MODEL_NAME = 'all-MiniLM-L6-v2'

# Anzahl Dateien zum Verarbeiten (None = alle)
MAX_FILES = None  # Setze auf z.B. 100 fuer Test

In [ ]:
# -------------------------
# JSON-Dateien einlesen
# -------------------------
def load_dlk_files(data_path, max_files=None):
    """Ladt alle JSON-Dateien aus dem DLK-Ordner"""
    json_files = glob.glob(os.path.join(data_path, '*.json'))
    
    if max_files:
        json_files = json_files[:max_files]
    
    poems = []
    for filepath in tqdm(json_files, desc='Lade JSON-Dateien'):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extrahiere Gedicht-Daten
            for poem_id, poem_data in data.items():
                metadata = poem_data.get('metadata', {})
                poem_content = poem_data.get('poem', {})
                
                # Extrahiere alle Zeilen
                lines = []
                for stanza_data in poem_content.values():
                    if isinstance(stanza_data, dict):
                        for line_data in stanza_data.values():
                            if isinstance(line_data, dict) and 'text' in line_data:
                                text = line_data['text'].strip()
                                if text:
                                    lines.append(text)
                
                if lines:
                    full_text = ' '.join(lines)
                    poems.append({
                        'id': poem_id,
                        'author': metadata.get('author', {}).get('name', 'Unbekannt'),
                        'title': metadata.get('title', 'Ohne Titel'),
                        'year': metadata.get('pub_year', 'Unbekannt'),
                        'lines': lines,
                        'full_text': full_text,
                        'num_lines': len(lines)
                    })
        except Exception as e:
            print(f'Fehler beim Laden von {filepath}: {e}')
    
    return poems

# Lade die Gedichte
poems = load_dlk_files(DLK_PATH, max_files=MAX_FILES)
print(f'Geladen: {len(poems)} Gedichte')

In [ ]:
# -------------------------
# Daten vorbereiten: Gedichte in kleine Dokumente splitten
# -------------------------
# Kleinere Chunks liefern oft bessere Treffer als ganze Gedichte,
# weil die Suche naeher am gesuchten Motiv bleibt.
CHUNK_SIZE = 8     # Zeilen pro Chunk
CHUNK_OVERLAP = 2  # Zeilen Ueberlappung fuer Kontext

def split_lines(lines, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Splittet Gedichtzeilen in ueberlappende Chunks."""
    step = chunk_size - overlap
    for start in range(0, len(lines), step):
        chunk = lines[start:start + chunk_size]
        if chunk:
            yield start, ' '.join(chunk)

documents = []
ids = []
metadatas = []

for poem in poems:
    for chunk_index, (start_line, chunk_text) in enumerate(split_lines(poem['lines'])):
        documents.append(chunk_text)
        ids.append(f"{poem['id']}_chunk_{chunk_index}")
        metadatas.append({
            'poem_id': poem['id'],
            'author': poem['author'],
            'title': poem['title'],
            'year': poem['year'],
            'chunk_index': chunk_index,
            'start_line': start_line + 1,
            'num_lines': poem['num_lines']
        })

print(f'Gedichte: {len(poems)}')
print(f'Dokument-Chunks: {len(documents)}')
print(f'Erster Chunk: {documents[0][:100]}...')

In [ ]:
# -------------------------
# Chroma DB initialisieren
# -------------------------
# Verwende PersistentClient fuer dauerhafte Speicherung
client = chromadb.PersistentClient(path=CHROMA_PATH)

# Collection mit Embedding-Funktion erstellen
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name=MODEL_NAME
    )
)

print(f'Chroma Collection "{COLLECTION_NAME}" bereit. Aktuelle Dokumentanzahl: {collection.count()}')

In [ ]:
# -------------------------
# Embedden und in Chroma speichern
# -------------------------
# Speichere in Batches um Speicherprobleme zu vermeiden
batch_size = 100

for i in tqdm(range(0, len(documents), batch_size), desc='Speichere in Chroma'):
    batch_docs = documents[i:i + batch_size]
    batch_ids = ids[i:i + batch_size]
    batch_metas = metadatas[i:i + batch_size]
    
    collection.add(
        documents=batch_docs,
        ids=batch_ids,
        metadatas=batch_metas
    )

print(f'Gespeichert: {collection.count()} Dokumente')

In [ ]:
# -------------------------
# Test-Query
# -------------------------
# Beispiel-Suchen
queries = [
    'Von Eise befreit sind Gewässer und Bachläufe'
]

for query in queries:
    print(f'\n{"="*60}')
    print(f'Query: {query}')
    print('{"-"*120}')
    
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    
    for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
        print(f'{i+1}. [{meta["author"]} - {meta["title"]}]')
        print(f'   {doc[:150]}...')
        print()

In [ ]:
# -------------------------
# Statistik
# -------------------------
print(f'Gesamtzahl Gedichte: {len(poems)}')
print(f'Gesamtzahl Dokumente in Chroma: {collection.count()}')

# Autor-Statistik
from collections import Counter
author_counts = Counter([p['author'] for p in poems])
print('\nTop 10 Autoren nach Gedichtanzahl:')
for author, count in author_counts.most_common(10):
    print(f'  {author}: {count} Gedichte')